# Decision Tree Extractor

**[NOTE]** This needs to be done on colab, so you will need gemini_prompter.py for the function.

This only works on colab because its a colab only api that doesn't require an api key.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.listdir()

['.config', 'drive', 'sample_data']

In [3]:
# You may need to adjust this based on where your datasets/paths are
# Make sure to work in the NLP Group Folder so its easier
main_path = "./drive/MyDrive/Shortcuts/NLP Group"
os.chdir(main_path)

In [4]:
os.listdir()

['Project2026.gdoc',
 'Final_project_template.ipynb',
 'Summary of Paper.gdoc',
 'NLP Project Plan.gdoc',
 'data_sets',
 'open_router',
 'Gemini_2-5-Flash',
 'DS8008_NLP_Project_DraftCode.ipynb']

In [ ]:
data_sets = os.listdir("./data_sets")

In [ ]:
data_sets

['house_votes_84',
 'boxing1',
 'acl',
 'heart_h',
 'boxing2',
 'colic',
 'hepatitis',
 'creditscore',
 'bankruptcy',
 'irish',
 'labor',
 'penguins',
 'japansolvent',
 'posttrauma',
 'vote']

## Initial Step of Generating 5 dt trees/functions for each dataset

In [ ]:
from google.colab import ai
from src.gemini_prompter import generate_dt_funcs

In [ ]:
subset = os.listdir("./data_sets")

In [ ]:
generate_dt_funcs(
    datasets_path="./data_sets",
    output_path="./Gemini_2-5-Flash",
    model=ai,
    num_dt=5
)

Generating DTs for each dataset:   0%|          | 0/5 [00:00<?, ?it/s]


 Error code: 429 - {'message': 'The estimated cost of this operation ($0.1640647) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}

 Error code: 429 - {'message': 'The estimated cost of this operation ($0.1640647) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}

 Error code: 429 - {'message': 'The estimated cost of this operation ($0.1640647) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}

 Error code: 429 - {'message': 'The estimated cost of this operation ($0.1640647) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}

 Error code: 429 - {'message': 'The estimated cost of this operation ($0.1640647) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


Generating DTs for each dataset:  20%|██        | 1/5 [00:11<00:45, 11.32s/it]


 Error code: 429 - {'message': 'The estimated cost of this operation ($0.1640014) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}

 Error code: 429 - {'message': 'The estimated cost of this operation ($0.1640014) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


Generating DTs for each dataset:  20%|██        | 1/5 [00:16<01:05, 16.45s/it]


KeyboardInterrupt: 

## Using Open Router (Older method)

This was the the older method we also tried using Open Router's APIs
* we decided not to continue this as it was taking too long.

https://openrouter.ai/google/gemma-3-27b-it:free/api

https://openrouter.ai/meta-llama/llama-3.3-70b-instruct:free

https://openrouter.ai/qwen/qwen3.6-plus:free

https://openrouter.ai/z-ai/glm-4.5-air:free/api


In [ ]:
from getpass import getpass
from openai import OpenAI

def generate_dt_funcs_open_router(datasets_path, output_path, models: list, use_subset=None):
  api_key = getpass("Enter your API key: ")
  data_sets = os.listdir(datasets_path)

  # sometimes you may only want to run it on a subset of all datasets
  if use_subset is not None:
    data_sets = [dataset for dataset in data_sets if dataset in use_subset]

  dt_main_path = Path(output_path)
  dt_main_path.mkdir(exist_ok=True)

  SYSTEM_PROMPT = f"""
    You are a domain expert with years of experience in building the best-performing decision trees.
    You have an astounding ability to identify the best features for the task at hand, and you know how to combine them to get the best predictions.
    Impressively, your profound world knowledge allows you to do that without looking at any real-world data.

    # Do the following Steps
    ### STEP 1
    I want you to induce a decision tree classifier based on features and a prediction target.
    I first give an example of the decision tree. Given Features and a new prediction target, I then want you to build a decision tree using the most important features.

    ### STEP 2
    Format the decision tree as a Python function that returns a single prediction as well as a list representing the truth values of the inner nodes.
    The entries of this list should be 1 if the condition of the corresponding inner node is satisfied, and 0 otherwise.
    Use only the feature names that I provide, generate the decision tree without training it on actual data, and return the Python function.

    ### OUTPUT INSTRUCTION
    Only return the decision tree function. Do not add any reasoning or other materials.
    """
  client = OpenAI(
      base_url = "https://openrouter.ai/api/v1",
      api_key=api_key
  )

  for dataset in tqdm(data_sets, desc="Generating DTs for each dataset"):
    # Read dataset description
    with open(os.path.join(datasets_path, dataset, 'description.txt')) as f:
      description_part = f.read()

    # Read specific prompt for dataset
    with open(os.path.join(datasets_path, dataset, 'prompt.txt')) as f:
      final_part = f.read()



    current_prompt = f"""
    ### DESCRIPTION OF DATASET
    {description_part}

    ### ADDITIONAL INSTRUCTIONS
    {final_part}
    """

    for model_name in models:
      try:
        completion = client.chat.completions.create(
            model=models[model_name],
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": current_prompt}
            ]
        )
        response = completion.choices[0].message.content
      except Exception as e:
        response = str(e)
        print(f"\n {response}")
      dataset_path = dt_main_path / dataset
      dataset_path.mkdir(exist_ok=True)
      response_path = dataset_path / f"{dataset}_{model_name}.txt"
      with open(response_path, 'w') as f:
        f.write(response)

In [120]:
models = {
    "Qwen3.6_Plus": "qwen/qwen3.6-plus:free",
    "Nvidia_nemotron-3": "nvidia/nemotron-3-super-120b-a12b:free",
    "GLM": "z-ai/glm-4.5-air:free"
    }

In [ ]:
generate_dt_funcs_open_router(
    datasets_path="./data_sets",
    output_path="./open_router",
    models=models
)